In [1]:
from pxd.datasets.hdf5.directory import HDF5DirectoryDataset, HDF5DirectoryDatasetConfig
import numpy as np
import pandas as pd
from typing import Dict, Any, Tuple, List,Optional
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from samples.sample import Sample
import os
import json
import pickle

In [2]:
def get_samples(data_path):
    config = HDF5DirectoryDatasetConfig(
        path=data_path
    )

    dataset = HDF5DirectoryDataset(config)

    dataset_dict = dataset.load()

    samples = {}
    for i in tqdm(list(dataset_dict.keys())):
        samples[i] = dataset_dict[i]()
    
    return samples

In [3]:
test_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/test_predictions"
test_pred_samples = get_samples(test_pred_path)

test_split_path = "/flux/vault/stator/03_splitting/stator_splitting_ds3_jul13/test/"
test_split_samples = get_samples(test_split_path)

100%|██████████| 100/100 [00:02<00:00, 34.96it/s]


In [4]:
import numpy as np

def _ensure_1d(arr, name="array"):
    """Convert input to a 1D numpy array."""
    arr = np.asarray(arr, dtype=float)

    if arr.ndim == 2 and 1 in arr.shape:
        arr = arr.reshape(-1)

    if arr.ndim != 1:
        raise ValueError(
            f"Expected a 1D array for {name}, got shape {arr.shape}"
        )

    return arr


def _ensure_2d(arr, name="array"):
    """Convert input to a 2D numpy array."""
    arr = np.asarray(arr, dtype=float)

    if arr.ndim != 2:
        raise ValueError(
            f"Expected a 2D array for {name}, got shape {arr.shape}"
        )

    return arr

In [5]:
def _get_first_available_key(dictionary, key_candidates, sample_id):
    for key in key_candidates:
        if key in dictionary:
            return dictionary[key]

    raise KeyError(
        f"None of the expected keys were found for sample '{sample_id}'. "
        f"Tried keys: {key_candidates}"
    )

In [6]:
def reconstruct_symmetric_matrix_from_upper_triangle(
    coeffs: Any,
    n: int = 16,
    diagonal_values: Optional[np.ndarray] = None,
    diagonal_strategy: str = "negative_row_sum"
) -> np.ndarray:
    """
    Reconstruct a full symmetric matrix from upper-triangle coefficients.

    Assumes coeffs were originally extracted using:
        a_matrix[np.triu_indices(n, k=1)]

    Parameters
    ----------
    coeffs : array-like
        Upper triangle coefficients excluding diagonal.
        Expected length = n * (n - 1) / 2.

    n : int
        Matrix size.

    diagonal_values : np.ndarray or None
        True diagonal values, if available.

    diagonal_strategy : str
        Options:
        - "from_true": use diagonal_values from the original a_matrix
        - "zero": set diagonal to zero
        - "negative_row_sum": set A_ii = -sum(A_ij for j != i)

    Returns
    -------
    a_matrix_pred : np.ndarray
        Reconstructed symmetric matrix, shape = (n, n)
    """
    coeffs = _ensure_1d(coeffs, name="a_matrix_coefficients_pred")

    expected_len = n * (n - 1) // 2
    if coeffs.shape[0] != expected_len:
        raise ValueError(
            f"Expected {expected_len} coefficients for upper triangle excluding diagonal, "
            f"got {coeffs.shape[0]}"
        )

    a_matrix_pred = np.zeros((n, n), dtype=float)

    upper_idx = np.triu_indices(n, k=1)

    a_matrix_pred[upper_idx] = coeffs
    a_matrix_pred = a_matrix_pred + a_matrix_pred.T

    if diagonal_strategy == "from_true":
        if diagonal_values is None:
            raise ValueError(
                "diagonal_strategy='from_true' requires diagonal_values from original a_matrix."
            )
        np.fill_diagonal(a_matrix_pred, diagonal_values)

    elif diagonal_strategy == "zero":
        np.fill_diagonal(a_matrix_pred, 0.0)

    elif diagonal_strategy == "negative_row_sum":
        diagonal = -np.sum(a_matrix_pred, axis=1)
        np.fill_diagonal(a_matrix_pred, diagonal)

    else:
        raise ValueError(
            "diagonal_strategy must be one of: 'from_true', 'zero', 'negative_row_sum'"
        )

    return a_matrix_pred

In [7]:
def calculate_metrics(y_true: Any, y_pred: Any, prefix: str = "") -> Dict[str, float]:
    """
    Calculate common regression metrics.

    Returns MAE, RMSE, relative RMSE, max absolute error, mean absolute percentage error,
    and R2 score.
    """
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"Shape mismatch for metrics: y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    error = y_pred - y_true
    abs_error = np.abs(error)

    mae = float(np.mean(abs_error))
    mse = float(np.mean(error ** 2))
    rmse = float(np.sqrt(mse))
    max_abs_error = float(np.max(abs_error))

    denominator = np.maximum(np.abs(y_true), 1e-12)
    mape = float(np.mean(abs_error / denominator) * 100)

    true_norm = float(np.linalg.norm(y_true))
    error_norm = float(np.linalg.norm(error))
    relative_l2_error = float(error_norm / max(true_norm, 1e-12))

    ss_res = float(np.sum(error ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else np.nan

    return {
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}relative_l2_error": relative_l2_error,
        f"{prefix}max_abs_error": max_abs_error,
        f"{prefix}mape_percent": mape,
        f"{prefix}r2": r2,
    }


def calculate_matrix_metrics(
    a_matrix_true: np.ndarray,
    a_matrix_pred: np.ndarray,
    prefix: str = "a_matrix_"
) -> Dict[str, float]:
    """
    Matrix-level metrics based on Frobenius norm and element-wise errors.
    """
    a_matrix_true = _ensure_2d(a_matrix_true, name="a_matrix_true")
    a_matrix_pred = _ensure_2d(a_matrix_pred, name="a_matrix_pred")

    if a_matrix_true.shape != a_matrix_pred.shape:
        raise ValueError(
            f"Shape mismatch: a_matrix_true={a_matrix_true.shape}, "
            f"a_matrix_pred={a_matrix_pred.shape}"
        )

    diff = a_matrix_pred - a_matrix_true

    fro_error = float(np.linalg.norm(diff, ord="fro"))
    fro_true = float(np.linalg.norm(a_matrix_true, ord="fro"))
    relative_fro_error = float(fro_error / max(fro_true, 1e-12))

    symmetry_error_pred = float(
        np.linalg.norm(a_matrix_pred - a_matrix_pred.T, ord="fro")
    )

    symmetry_error_true = float(
        np.linalg.norm(a_matrix_true - a_matrix_true.T, ord="fro")
    )

    metrics = {
        f"{prefix}frobenius_error": fro_error,
        f"{prefix}relative_frobenius_error": relative_fro_error,
        f"{prefix}symmetry_error_true": symmetry_error_true,
        f"{prefix}symmetry_error_pred": symmetry_error_pred,
    }

    metrics.update(
        calculate_metrics(
            a_matrix_true,
            a_matrix_pred,
            prefix=f"{prefix}elementwise_"
        )
    )

    return metrics


def build_coefficient_dataframe(
    sample_id: str,
    a_matrix_coefficients: np.ndarray,
    a_matrix_coefficients_pred: np.ndarray,
    n: int = 16
) -> pd.DataFrame:
    """
    Create long-format coefficient table for one sample.

    Each row corresponds to one upper-triangle matrix coefficient.
    """
    upper_i, upper_j = np.triu_indices(n, k=1)

    df = pd.DataFrame({
        "sample_id": sample_id,
        "matrix_i": upper_i,
        "matrix_j": upper_j,
        "a_matrix_coefficient_true": a_matrix_coefficients,
        "a_matrix_coefficient_pred": a_matrix_coefficients_pred,
    })

    df["a_matrix_coefficient_error"] = (
        df["a_matrix_coefficient_pred"] - df["a_matrix_coefficient_true"]
    )

    df["a_matrix_coefficient_abs_error"] = np.abs(
        df["a_matrix_coefficient_error"]
    )

    return df

## Modified Function to derive Q-Matrix-Pred for Polynomial Regression Ridge

In [8]:
def build_and_evaluate_heatflux_predictions_from_two_sources(
    test_pred_samples: Dict[str, Dict[str, Any]],
    test_split_samples: Dict[str, Dict[str, Any]],
    n: int = 16,
    output_dir: str = "heatflux_evaluation_output_two_sources",
    diagonal_strategy: str = "negative_row_sum",
    strict_sample_match: bool = False
) -> Tuple[Dict[str, Dict[str, np.ndarray]], pd.DataFrame, pd.DataFrame]:
    """
    Build and evaluate heat flux predictions by combining data from two sources.

    Source 1:
        test_pred_samples
        Contains predicted upper-triangle a_matrix coefficients:
            a_matrix_coefficients_pred

    Source 2:
        test_split_samples
        Contains true simulation data:
            t_matrix
            q_matrix

        Optionally may contain:
            a_matrix
            a_matrix_coefficients

    Workflow per sample:
        1. Match sample_id between test_pred_samples and test_split_samples
        2. Extract a_matrix_coefficients_pred from test_pred_samples
        3. Extract t_matrix and q_matrix from test_split_samples
        4. Reconstruct a_matrix_pred using negative_row_sum strategy
        5. Calculate q_pred = a_matrix_pred @ t_matrix
        6. Evaluate q_pred against q_matrix
        7. Save results to pickle, CSV, and NPZ

    Parameters
    ----------
    test_pred_samples : dict
        Dictionary containing predicted a_matrix coefficients mapped by sample_id.

    test_split_samples : dict
        Dictionary containing t_matrix and q_matrix mapped by sample_id.

    n : int, default=16
        Size of the square a_matrix.

    output_dir : str
        Directory where outputs will be saved.

    diagonal_strategy : str, default="negative_row_sum"
        Strategy for reconstructing the diagonal of a_matrix_pred.
        Recommended for this case: "negative_row_sum".

    strict_sample_match : bool, default=False
        If True, raise an error when both dictionaries do not contain exactly
        the same sample IDs.
        If False, process only sample IDs present in both dictionaries.

    Returns
    -------
    per_sample_results : dict
        Nested dictionary containing extracted, reconstructed, and predicted arrays.

    metrics_df : pd.DataFrame
        One row per sample plus a GLOBAL row with aggregated metrics.

    coefficients_df : pd.DataFrame
        Long-format dataframe of predicted coefficients, and true coefficients
        if available.
    """

    os.makedirs(output_dir, exist_ok=True)

    expected_coeff_len = n * (n - 1) // 2

    pred_sample_ids = set(test_pred_samples.keys())
    split_sample_ids = set(test_split_samples.keys())

    common_sample_ids = sorted(pred_sample_ids.intersection(split_sample_ids))

    missing_in_pred = sorted(split_sample_ids - pred_sample_ids)
    missing_in_split = sorted(pred_sample_ids - split_sample_ids)

    if strict_sample_match:
        if missing_in_pred or missing_in_split:
            raise ValueError(
                "Sample ID mismatch between test_pred_samples and test_split_samples.\n"
                f"Samples missing in test_pred_samples: {missing_in_pred}\n"
                f"Samples missing in test_split_samples: {missing_in_split}"
            )

    if len(common_sample_ids) == 0:
        raise ValueError(
            "No common sample IDs found between test_pred_samples and test_split_samples."
        )

    print(f"Number of samples in test_pred_samples:  {len(pred_sample_ids)}")
    print(f"Number of samples in test_split_samples: {len(split_sample_ids)}")
    print(f"Number of common samples processed:      {len(common_sample_ids)}")

    if missing_in_pred:
        print(
            f"Warning: {len(missing_in_pred)} samples exist in test_split_samples "
            "but not in test_pred_samples."
        )

    if missing_in_split:
        print(
            f"Warning: {len(missing_in_split)} samples exist in test_pred_samples "
            "but not in test_split_samples."
        )

    per_sample_results: Dict[str, Dict[str, np.ndarray]] = {}
    metrics_records: List[Dict[str, Any]] = []
    coefficient_dfs: List[pd.DataFrame] = []

    upper_i, upper_j = np.triu_indices(n, k=1)

    for sample_id in common_sample_ids:

        pred_payload = test_pred_samples[sample_id]
        split_payload = test_split_samples[sample_id]

        # --------------------------------------------------
        # Extract heatflux blocks from both dictionaries
        # --------------------------------------------------

        try:
            pred_heatflux = pred_payload["data"]["heatflux_matrix"]
        except KeyError as e:
            raise KeyError(
                f"Missing predicted heatflux block for sample '{sample_id}'. "
                f"Expected test_pred_samples[sample_id]['data']['heatflux_matrix']. "
                f"Original error: {e}"
            ) from e

        try:
            split_heatflux = split_payload["data"]["heatflux_matrix"]
        except KeyError as e:
            raise KeyError(
                f"Missing split heatflux block for sample '{sample_id}'. "
                f"Expected test_split_samples[sample_id]['data']['heatflux_matrix']. "
                f"Original error: {e}"
            ) from e

        # --------------------------------------------------
        # Extract predicted coefficients from test_pred_samples
        # --------------------------------------------------

        a_matrix_coefficients_pred = _ensure_1d(
            _get_first_available_key(
                pred_heatflux,
                [
                    "a_matrix_coefficients_pred",
                    "a_matrix_coefficient_pred",
                    "a_matrix_coefficients_prediction",
                    "a_matrix_coefficient_prediction",
                    "a_matrix_pred_coefficients",
                    "a_matrix_pred_coefficient"
                ],
                sample_id
            ),
            name=f"{sample_id}.a_matrix_coefficients_pred"
        )

        # --------------------------------------------------
        # Extract t_matrix and q_matrix from test_split_samples
        # --------------------------------------------------

        t_matrix = _ensure_2d(
            _get_first_available_key(
                split_heatflux,
                [
                    "t_matrix",
                    "T_matrix",
                    "temperature_matrix"
                ],
                sample_id
            ),
            name=f"{sample_id}.t_matrix"
        )

        q_matrix = _ensure_2d(
            _get_first_available_key(
                split_heatflux,
                [
                    "q_matrix",
                    "Q_matrix",
                    "heatflux_matrix_q"
                ],
                sample_id
            ),
            name=f"{sample_id}.q_matrix"
        )

        # --------------------------------------------------
        # Optional true a_matrix and true coefficients
        # These are useful for extra evaluation if available
        # --------------------------------------------------

        a_matrix = None
        a_matrix_coefficients = None

        try:
            a_matrix = _ensure_2d(
                _get_first_available_key(
                    split_heatflux,
                    [
                        "a_matrix",
                        "A_matrix",
                        "response_matrix"
                    ],
                    sample_id
                ),
                name=f"{sample_id}.a_matrix"
            )
        except KeyError:
            a_matrix = None

        try:
            a_matrix_coefficients = _ensure_1d(
                _get_first_available_key(
                    split_heatflux,
                    [
                        "a_matrix_coefficients",
                        "a_matrix_coefficient",
                        "a_matrix_coeffiecients"
                    ],
                    sample_id
                ),
                name=f"{sample_id}.a_matrix_coefficients"
            )
        except KeyError:
            a_matrix_coefficients = None

        # --------------------------------------------------
        # Validate shapes
        # --------------------------------------------------

        if a_matrix_coefficients_pred.shape[0] != expected_coeff_len:
            raise ValueError(
                f"Expected {expected_coeff_len} predicted coefficients for sample "
                f"'{sample_id}', got {a_matrix_coefficients_pred.shape[0]}."
            )

        if t_matrix.shape[0] != n:
            raise ValueError(
                f"Expected t_matrix to have {n} rows for sample '{sample_id}', "
                f"got shape {t_matrix.shape}."
            )

        if q_matrix.shape[0] != n:
            raise ValueError(
                f"Expected q_matrix to have {n} rows for sample '{sample_id}', "
                f"got shape {q_matrix.shape}."
            )

        if q_matrix.shape[1] != t_matrix.shape[1]:
            raise ValueError(
                f"q_matrix and t_matrix must have the same number of columns for sample "
                f"'{sample_id}'. Got q_matrix shape {q_matrix.shape} and "
                f"t_matrix shape {t_matrix.shape}."
            )

        if a_matrix is not None and a_matrix.shape != (n, n):
            raise ValueError(
                f"Expected a_matrix shape ({n}, {n}) for sample '{sample_id}', "
                f"got shape {a_matrix.shape}."
            )

        if a_matrix_coefficients is not None:
            if a_matrix_coefficients.shape[0] != expected_coeff_len:
                raise ValueError(
                    f"Expected {expected_coeff_len} true coefficients for sample "
                    f"'{sample_id}', got {a_matrix_coefficients.shape[0]}."
                )

        # --------------------------------------------------
        # Reconstruct predicted A matrix
        # --------------------------------------------------
        # Since diagonal_strategy="negative_row_sum", no true diagonal is needed.
        # A_ii = -sum(A_ij for j != i)

        a_matrix_pred = reconstruct_symmetric_matrix_from_upper_triangle(
            coeffs=a_matrix_coefficients_pred,
            n=n,
            diagonal_values=None,
            diagonal_strategy=diagonal_strategy
        )

        # --------------------------------------------------
        # Calculate predicted heat flux
        # --------------------------------------------------

        q_pred = a_matrix_pred @ t_matrix

        if q_pred.shape != q_matrix.shape:
            raise ValueError(
                f"q_pred shape mismatch for sample '{sample_id}'. "
                f"q_pred shape = {q_pred.shape}, q_matrix shape = {q_matrix.shape}."
            )

        # --------------------------------------------------
        # Calculate errors
        # --------------------------------------------------

        q_matrix_error = q_pred - q_matrix

        a_matrix_error = None
        if a_matrix is not None:
            a_matrix_error = a_matrix_pred - a_matrix

        coefficient_error = None
        if a_matrix_coefficients is not None:
            coefficient_error = a_matrix_coefficients_pred - a_matrix_coefficients

        # --------------------------------------------------
        # Store arrays
        # --------------------------------------------------

        sample_result: Dict[str, np.ndarray] = {
            "a_matrix_coefficients_pred": a_matrix_coefficients_pred,
            "a_matrix_pred": a_matrix_pred,
            "t_matrix": t_matrix,
            "q_matrix": q_matrix,
            "q_pred": q_pred,
            "q_matrix_error": q_matrix_error,
        }

        if a_matrix is not None:
            sample_result["a_matrix"] = a_matrix
            sample_result["a_matrix_error"] = a_matrix_error

        if a_matrix_coefficients is not None:
            sample_result["a_matrix_coefficients"] = a_matrix_coefficients
            sample_result["a_matrix_coefficients_error"] = coefficient_error

        per_sample_results[sample_id] = sample_result

        # --------------------------------------------------
        # Metrics per sample
        # --------------------------------------------------

        sample_metrics: Dict[str, Any] = {
            "sample_id": sample_id,
            "n": n,
            "diagonal_strategy": diagonal_strategy,
            "num_coefficients": expected_coeff_len,
            "num_temperature_cases": t_matrix.shape[1],
            "has_true_a_matrix": a_matrix is not None,
            "has_true_coefficients": a_matrix_coefficients is not None,
        }

        # q_matrix metrics are the main metrics for this workflow
        sample_metrics.update(
            calculate_metrics(
                q_matrix,
                q_pred,
                prefix="q_matrix_"
            )
        )

        # Optional coefficient-level metrics
        if a_matrix_coefficients is not None:
            sample_metrics.update(
                calculate_metrics(
                    a_matrix_coefficients,
                    a_matrix_coefficients_pred,
                    prefix="coefficients_"
                )
            )

        # Optional full matrix metrics
        if a_matrix is not None:
            sample_metrics.update(
                calculate_matrix_metrics(
                    a_matrix,
                    a_matrix_pred,
                    prefix="a_matrix_"
                )
            )

        metrics_records.append(sample_metrics)

        # --------------------------------------------------
        # Coefficient long-format dataframe
        # --------------------------------------------------

        if a_matrix_coefficients is not None:
            coeff_df = build_coefficient_dataframe(
                sample_id=sample_id,
                a_matrix_coefficients=a_matrix_coefficients,
                a_matrix_coefficients_pred=a_matrix_coefficients_pred,
                n=n
            )
        else:
            coeff_df = pd.DataFrame({
                "sample_id": sample_id,
                "matrix_i": upper_i,
                "matrix_j": upper_j,
                "a_matrix_coefficient_pred": a_matrix_coefficients_pred,
            })

        coefficient_dfs.append(coeff_df)

    # --------------------------------------------------
    # Build metrics dataframe
    # --------------------------------------------------

    metrics_df = pd.DataFrame(metrics_records)

    if coefficient_dfs:
        coefficients_df = pd.concat(coefficient_dfs, ignore_index=True)
    else:
        coefficients_df = pd.DataFrame()

    # --------------------------------------------------
    # Global metrics across all processed samples
    # --------------------------------------------------

    all_q_true = np.concatenate([
        result["q_matrix"].reshape(-1)
        for result in per_sample_results.values()
    ])

    all_q_pred = np.concatenate([
        result["q_pred"].reshape(-1)
        for result in per_sample_results.values()
    ])

    global_metrics: Dict[str, Any] = {
        "sample_id": "GLOBAL",
        "n": n,
        "diagonal_strategy": diagonal_strategy,
        "num_coefficients": len(per_sample_results) * expected_coeff_len,
        "num_temperature_cases": np.nan,
        "has_true_a_matrix": all(
            "a_matrix" in result for result in per_sample_results.values()
        ),
        "has_true_coefficients": all(
            "a_matrix_coefficients" in result
            for result in per_sample_results.values()
        ),
    }

    global_metrics.update(
        calculate_metrics(
            all_q_true,
            all_q_pred,
            prefix="q_matrix_"
        )
    )

    # Optional global coefficient metrics
    if global_metrics["has_true_coefficients"]:
        all_coeff_true = np.concatenate([
            result["a_matrix_coefficients"]
            for result in per_sample_results.values()
        ])

        all_coeff_pred = np.concatenate([
            result["a_matrix_coefficients_pred"]
            for result in per_sample_results.values()
        ])

        global_metrics.update(
            calculate_metrics(
                all_coeff_true,
                all_coeff_pred,
                prefix="coefficients_"
            )
        )

    # Optional global a_matrix metrics
    if global_metrics["has_true_a_matrix"]:
        all_a_true = np.concatenate([
            result["a_matrix"].reshape(-1)
            for result in per_sample_results.values()
        ])

        all_a_pred = np.concatenate([
            result["a_matrix_pred"].reshape(-1)
            for result in per_sample_results.values()
        ])

        global_metrics.update(
            calculate_metrics(
                all_a_true,
                all_a_pred,
                prefix="a_matrix_elementwise_"
            )
        )

    metrics_df = pd.concat(
        [metrics_df, pd.DataFrame([global_metrics])],
        ignore_index=True
    )

    # --------------------------------------------------
    # Save outputs
    # --------------------------------------------------

    pickle_path = os.path.join(output_dir, "per_sample_results.pkl")
    with open(pickle_path, "wb") as f:
        pickle.dump(per_sample_results, f)

    metrics_csv_path = os.path.join(output_dir, "evaluation_metrics.csv")
    metrics_df.to_csv(metrics_csv_path, index=False)

    coefficients_csv_path = os.path.join(
        output_dir,
        "coefficient_predictions_long_format.csv"
    )
    coefficients_df.to_csv(coefficients_csv_path, index=False)

    npz_path = os.path.join(output_dir, "per_sample_arrays.npz")

    npz_save_dict = {}

    for sample_id, result in per_sample_results.items():
        safe_sample_id = (
            str(sample_id)
            .replace("/", "_")
            .replace("\\", "_")
            .replace(" ", "_")
            .replace(":", "_")
        )

        for key, value in result.items():
            if value is not None:
                npz_save_dict[f"{safe_sample_id}__{key}"] = value

    np.savez_compressed(npz_path, **npz_save_dict)

    print("Saved outputs:")
    print(f"  Pickle arrays:      {pickle_path}")
    print(f"  Metrics CSV:        {metrics_csv_path}")
    print(f"  Coefficients CSV:   {coefficients_csv_path}")
    print(f"  Compressed NPZ:     {npz_path}")

    return per_sample_results, metrics_df, coefficients_df

In [ ]:
per_sample_results, metrics_df, coefficients_df = build_and_evaluate_heatflux_predictions_from_two_sources(
    test_pred_samples=test_pred_samples,
    test_split_samples=test_split_samples,
    n=16,
    output_dir="output/poly_reg_test",
    diagonal_strategy="negative_row_sum",
    strict_sample_match=False)

Number of samples in test_pred_samples:  100
Number of samples in test_split_samples: 100
Number of common samples processed:      100
Saved outputs:
  Pickle arrays:      heatflux_evaluation_output_two_sources/per_sample_results.pkl
  Metrics CSV:        heatflux_evaluation_output_two_sources/evaluation_metrics.csv
  Coefficients CSV:   heatflux_evaluation_output_two_sources/coefficient_predictions_long_format.csv
  Compressed NPZ:     heatflux_evaluation_output_two_sources/per_sample_arrays.npz


In [13]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)
        
with open("output/poly_reg_test/per_sample_results_test.json", "w") as f:
        json.dump(per_sample_results, f, sort_keys=True, cls=NumpyEncoder)